In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

In [ ]:
loader=PyPDFDirectoryLoader("./us_census")
documents=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
final_documents=text_splitter.split_documents(documents)
final_documents[0] 

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census/acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.\n1 Public policy changes included \nthe renewal of the Public Health Emergency, w

In [ ]:
import langchain
import langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

1.3.9
0.4.2


In [26]:
# HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5", # sentence-transformers/all-MiniLM-l6-v2
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15862.76it/s]


In [28]:
import numpy as np
np.array(embeddings.embed_query(final_documents[0].page_content)).shape

(384,)

In [29]:
## VectorStore Creation
vectorstore=FAISS.from_documents(final_documents[:120],embeddings)

In [ ]:
## Query using Similarity Search
query="WHAT IS HEALTH INSURANCE COVERAGE?"
relevant_docments=vectorstore.similarity_search(query)
print(relevant_docments[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [31]:
retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":3})
print(retriever)

tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3096cde90> search_kwargs={'k': 3}


In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_KEY"] = ""
HUGGINGFACEHUB_API_KEY=os.environ["HUGGINGFACEHUB_API_KEY"]

In [ ]:
import os
from langchain_huggingface import HuggingFaceEndpoint

# Replace this string with your NEW Fine-grained token
os.environ["HUGGINGFACEHUB_API_KEY"] = ""
HUGGINGFACEHUB_API_KEY = os.environ["HUGGINGFACEHUB_API_KEY"]

hf = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-v0.1",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_KEY,
    temperature=0.1,
    max_new_tokens=500
)

query = "What is the health insurance coverage?"
response = hf.invoke(query)

print(response)



The health insurance coverage is a contract between the insurer and the insured. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premium to the insurer.

What is the health insurance coverage?

The health insurance coverage is a contract between the insurer and the insured. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premium to the insurer.

What is the health insurance coverage?

The health insurance coverage is a contract between the insurer and the insured. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premium to the insurer.

What is the health insurance coverage?

The health insurance coverage is a contract between the insurer and the insured. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premium

In [49]:
llm=hf
llm.invoke(query)

'\n\nThe health insurance coverage is a contract between the insurer and the insured. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premiums to the insurer. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premiums to the insurer.\n\nWhat is the health insurance coverage?\n\nThe health insurance coverage is a contract between the insurer and the insured. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premiums to the insurer. The insurer agrees to pay the insured for the medical expenses incurred by the insured. The insured agrees to pay the premiums to the insurer.\n\nWhat is the health insurance coverage?\n\nThe health insurance coverage is a contract between the insurer and the insured. The insurer agrees to pay the insured for the medical expenses incurred by the insured. Th

In [50]:
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{question}

Helpful Answers:
 """

In [51]:
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])

In [52]:
retrievalQA=RetrievalQA.from_chain_type(
    llm=hf,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt":prompt}
)

In [53]:
query="""DIFFERENCES IN THE
UNINSURED RATE BY STATE
IN 2022"""

In [54]:
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})
print(result['result'])

JSONDecodeError: Expecting ',' delimiter: line 1 column 3151 (char 3150)